# K-EmoCon Dataset — Exploratory Data Analysis
Full multimodal EDA: annotations, physiological signals, audio, video, correlations, lagged cross-correlations, dyadic synchrony.

In [ ]:
import sys
sys.path.insert(0, '.')
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from scipy import stats
from scipy.stats import pearsonr, spearmanr

from config import DYADS, DATA_ROOT, DOWNLOADS
from data_loader import (load_metadata, get_start_time,
                          load_e4_all, load_neuro_all,
                          load_annotations, load_partner_annotations,
                          load_baseline_stats)
from features import build_segment_features
import audio_features as aud
import video_features as vid

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
ANNOT_ROOT = DATA_ROOT / 'emotion_annotations'
CACHE_AUD  = DATA_ROOT / 'audio_features_cache'
CACHE_VID  = DATA_ROOT / 'video_features_cache'

print('Setup complete')

In [ ]:
# ── Load all segment tables (physio + audio + video) ──────────────────────
subjects, avail = load_metadata()

seg_tables = {}
for pid in sorted(subjects['pid'].unique()):
    try:
        start_ms = get_start_time(pid, subjects)
        e4    = load_e4_all(pid, start_ms)
        neuro = load_neuro_all(pid, start_ms)
        annot = load_annotations(pid)
        if annot is None or len(annot) < 5:
            continue
        bl = load_baseline_stats(pid, subjects)
        seg = build_segment_features(pid, e4, neuro, annot, bl)

        # merge audio
        pid_a, pid_b = next(((a,b) for a,b in DYADS if a==pid or b==pid), (None,None))
        if pid_a and pid_b:
            da, db = aud.load_or_extract(pid_a, pid_b, len(seg), len(seg))
            aud_df = da if pid == pid_a else db
            seg = aud.merge_audio_into_seg_table(seg, aud_df)

        # merge video
        vcache = CACHE_VID / f'P{pid}.csv'
        if vcache.exists():
            vdf = pd.read_csv(vcache)
            seg = vid.merge_video_into_seg_table(seg, vdf)

        seg_tables[pid] = seg
    except Exception as e:
        print(f'P{pid}: {e}')

# Standardise columns
all_cols = set(c for df in seg_tables.values() for c in df.columns)
meta = {'pid','seconds','arousal','valence'}
for pid, df in seg_tables.items():
    for c in all_cols - meta:
        if c not in df.columns:
            seg_tables[pid][c] = np.nan

# Master dataframe
master = pd.concat(seg_tables.values(), ignore_index=True)
feat_cols = sorted(all_cols - meta)
physio_cols = [c for c in feat_cols if c.startswith(('e4_','bw_','attention','meditation','polar'))]
audio_cols  = [c for c in feat_cols if c.startswith('aud_')]
video_cols  = [c for c in feat_cols if c.startswith('vid_')]

print(f'Participants loaded: {len(seg_tables)}')
print(f'Total segments: {len(master)}')
print(f'Features: {len(physio_cols)} physio | {len(audio_cols)} audio | {len(video_cols)} video')

## 1. Dataset Overview

In [ ]:
# Signal availability heatmap
avail_disp = avail.set_index('pid')[[
    'E4_ACC','E4_BVP','E4_EDA','E4_HR','E4_IBI','E4_TEMP',
    'Attention','BrainWave','Meditation','Polar_HR',
    'debate_audio','debate_recording',
    'aggregated_external_annotations','partner_annotations','self_annotations'
]].replace({'TRUE':1,'FALSE':0,True:1,False:0})

fig, ax = plt.subplots(figsize=(14, 9))
sns.heatmap(avail_disp.T.astype(float), ax=ax, cmap='YlGn',
            linewidths=0.3, linecolor='white',
            cbar_kws={'label':'Available'},
            xticklabels=avail_disp.index, yticklabels=True)
ax.set_xlabel('Participant ID')
ax.set_title('Signal & Annotation Availability per Participant', fontsize=14)
plt.tight_layout()
plt.show()

print('\nAvailability summary (% of participants):')
print((avail_disp.mean()*100).round(1).to_string())

In [ ]:
# Session durations
subjects['duration_sec'] = (subjects['endTime'] - subjects['startTime']) / 1000
subjects['baseline_sec'] = (subjects['startTime'] - subjects['initTime']) / 1000
segs_per_pid = master.groupby('pid')['seconds'].count().rename('n_segments')
overview = subjects[['pid','duration_sec','baseline_sec']].merge(segs_per_pid, on='pid', how='left')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, title in zip(axes,
    ['duration_sec','baseline_sec','n_segments'],
    ['Debate duration (s)','Baseline period (s)','Annotated segments']):
    overview[col].dropna().plot.hist(bins=16, ax=ax, color='steelblue', edgecolor='white')
    ax.axvline(overview[col].mean(), color='red', linestyle='--', label=f'mean={overview[col].mean():.0f}')
    ax.set_title(title); ax.legend()
plt.suptitle('Session Statistics (N=32)', fontsize=13)
plt.tight_layout()
plt.show()

## 2. Annotation Analysis

In [ ]:
# Load all annotation types for all participants
ext_all, par_all = [], []
for pid in sorted(seg_tables.keys()):
    ea = load_annotations(pid)
    if ea is not None:
        ea['pid'] = pid; ext_all.append(ea)
    pa = load_partner_annotations(pid)
    if pa is not None:
        pa['pid'] = pid; par_all.append(pa)

ext_df = pd.concat(ext_all, ignore_index=True)
par_df = pd.concat(par_all, ignore_index=True)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Distributions
for i, (df, label) in enumerate([(ext_df,'External Observer'),(par_df,'Partner')]):
    for j, tgt in enumerate(['arousal','valence']):
        ax = axes[i][j]
        ax.hist(df[tgt].dropna(), bins=20, color=['#4C72B0','#DD8452'][i],
                edgecolor='white', alpha=0.85)
        ax.set_title(f'{label} — {tgt}')
        ax.set_xlabel('Score'); ax.set_ylabel('Count')
        m, s = df[tgt].mean(), df[tgt].std()
        ax.axvline(m, color='red', linestyle='--', label=f'μ={m:.2f}, σ={s:.2f}')
        ax.legend(fontsize=8)

# Arousal vs valence scatter (external)
ax = axes[0][2]
ax.hexbin(ext_df['valence'], ext_df['arousal'], gridsize=20, cmap='Blues')
ax.set_xlabel('Valence'); ax.set_ylabel('Arousal')
ax.set_title('External: Arousal–Valence space')

# External vs partner correlation
ax = axes[1][2]
merged_ap = ext_df.merge(par_df, on=['pid','seconds'], suffixes=('_ext','_par'))
for tgt, color in [('arousal','#4C72B0'),('valence','#DD8452')]:
    r, p = pearsonr(merged_ap[f'{tgt}_ext'].dropna(),
                    merged_ap[f'{tgt}_par'].dropna())
    ax.scatter(merged_ap[f'{tgt}_ext'], merged_ap[f'{tgt}_par'],
               alpha=0.05, s=5, color=color, label=f'{tgt} r={r:.2f}')
ax.plot([1,5],[1,5], 'k--', linewidth=0.8)
ax.set_xlabel('External'); ax.set_ylabel('Partner')
ax.set_title('External vs Partner agreement')
ax.legend(fontsize=9)

plt.suptitle('Emotion Annotation Analysis', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Inter-annotator reliability (ICC approximation via pairwise Pearson)
indiv_dir = ANNOT_ROOT / 'external_annotations'
icc_rows = []
for pid in sorted(seg_tables.keys()):
    raters = []
    for r in range(1, 6):
        p = indiv_dir / f'P{pid}.R{r}.csv'
        if p.exists():
            df = pd.read_csv(p)[['seconds','arousal','valence']]
            df.columns = ['seconds', f'ar_R{r}', f'va_R{r}']
            raters.append(df.set_index('seconds'))
    if len(raters) < 2:
        continue
    combined = pd.concat(raters, axis=1)
    for tgt_prefix, tgt in [('ar_','arousal'),('va_','valence')]:
        cols = [c for c in combined.columns if c.startswith(tgt_prefix)]
        mat = combined[cols].dropna()
        if len(mat) < 10:
            continue
        rs = []
        for i in range(len(cols)):
            for j in range(i+1, len(cols)):
                r, _ = pearsonr(mat.iloc[:,i], mat.iloc[:,j])
                rs.append(r)
        icc_rows.append({'pid':pid, 'target':tgt,
                          'mean_pairwise_r': np.mean(rs),
                          'n_raters': len(cols)})

icc_df = pd.DataFrame(icc_rows)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, tgt in zip(axes, ['arousal','valence']):
    sub = icc_df[icc_df['target']==tgt].sort_values('pid')
    ax.bar(sub['pid'], sub['mean_pairwise_r'], color='steelblue', alpha=0.8)
    ax.axhline(sub['mean_pairwise_r'].mean(), color='red', linestyle='--',
               label=f'mean r = {sub["mean_pairwise_r"].mean():.2f}')
    ax.set_xlabel('Participant ID')
    ax.set_ylabel('Mean pairwise Pearson r')
    ax.set_title(f'Inter-Rater Agreement ({tgt})')
    ax.legend()
    ax.set_ylim(0, 1)

plt.suptitle('Inter-Annotator Reliability (External Raters, pairwise Pearson r)', fontsize=13)
plt.tight_layout()
plt.show()
print('\nOverall mean pairwise r:')
print(icc_df.groupby('target')['mean_pairwise_r'].agg(['mean','std','min','max']).round(3))

In [ ]:
# Annotation time series — 4 sample sessions
sample_pids = [1, 5, 13, 25]
fig, axes = plt.subplots(len(sample_pids), 1, figsize=(14, 10), sharex=False)

for ax, pid in zip(axes, sample_pids):
    ea = load_annotations(pid)
    pa = load_partner_annotations(pid)
    if ea is not None:
        ax.plot(ea['seconds'], ea['arousal'], label='Arousal (ext)', color='#4C72B0')
        ax.plot(ea['seconds'], ea['valence'], label='Valence (ext)', color='#4C72B0',
                linestyle='--', alpha=0.7)
    if pa is not None:
        ax.plot(pa['seconds'], pa['arousal'], label='Arousal (partner)', color='#DD8452',
                alpha=0.6)
        ax.plot(pa['seconds'], pa['valence'], label='Valence (partner)', color='#DD8452',
                linestyle='--', alpha=0.4)
    ax.set_ylabel('Score'); ax.set_title(f'P{pid}')
    ax.set_ylim(0.5, 5.5); ax.legend(fontsize=7, ncol=4)

axes[-1].set_xlabel('Time (seconds into debate)')
plt.suptitle('Annotation Time Series — Sample Sessions', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Physiological Signals

In [ ]:
# Distribution of all physio features
phys_plot = [c for c in physio_cols if not c.startswith('bw_')]  # skip individual EEG bands
n = len(phys_plot)
ncols = 4
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 2.8))
axes_flat = axes.flatten()

for i, col in enumerate(phys_plot):
    data = master[col].dropna()
    axes_flat[i].hist(data, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes_flat[i].set_title(col.replace('_mean','\nmean').replace('_std','\nstd'), fontsize=8)
    axes_flat[i].set_ylabel('Count')

for j in range(i+1, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle('Physiological Feature Distributions (baseline-normalised)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# NaN rate heatmap per participant × feature
nan_mat = master.groupby('pid')[feat_cols].apply(lambda df: df.isna().mean())

fig, ax = plt.subplots(figsize=(18, 9))
sns.heatmap(nan_mat.T, ax=ax, cmap='Reds', vmin=0, vmax=1,
            linewidths=0.1, linecolor='white',
            xticklabels=nan_mat.index, yticklabels=True)
ax.set_xlabel('Participant ID')
ax.set_title('NaN Rate per Feature × Participant (red = missing)', fontsize=13)
plt.tight_layout()
plt.show()

# Overall NaN rate per feature
nan_overall = master[feat_cols].isna().mean().sort_values(ascending=False)
print('Top-10 features with most missing data:')
print(nan_overall.head(10).round(3).to_string())

In [ ]:
# Baseline period vs debate period: compare HR & EDA raw distributions
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
signals = [('E4_HR','e4_hr_mean','HR (bpm)'),('E4_EDA','e4_eda_mean','EDA (μS)')]

for row, (sig_name, feat_col, label) in enumerate(signals):
    baseline_vals, debate_vals = [], []
    for pid in sorted(seg_tables.keys()):
        start_ms = get_start_time(pid, subjects)
        from data_loader import _load_signal_raw, E4_DIR
        df = _load_signal_raw(E4_DIR / str(pid) / f'{sig_name}.csv', start_ms)
        if df is None or 'value' not in df.columns:
            continue
        baseline_vals.extend(df[df['t_sec'] < 0]['value'].dropna().tolist())
        debate_vals.extend(df[(df['t_sec'] >= 0) & (df['t_sec'] <= 900)]['value'].dropna().tolist())

    for col_idx, (vals, period_label, color) in enumerate([
        (baseline_vals, 'Baseline', '#4C72B0'),
        (debate_vals,   'Debate',   '#DD8452')
    ]):
        ax = axes[row][col_idx]
        ax.hist(vals, bins=50, color=color, edgecolor='white', alpha=0.8, density=True)
        ax.set_title(f'{label} — {period_label}')
        ax.set_xlabel(label)
        ax.set_ylabel('Density')
        ax.axvline(np.median(vals), color='red', linestyle='--',
                   label=f'median={np.median(vals):.1f}')
        ax.legend(fontsize=9)

plt.suptitle('Physiological Signals: Baseline vs Debate Period', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Sample physiological time series (one session)
pid_ex = 13   # well-covered participant
start_ms = get_start_time(pid_ex, subjects)
e4_ex = load_e4_all(pid_ex, start_ms)
annot_ex = load_annotations(pid_ex)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
signal_info = [
    ('E4_HR',   'value', 'HR (bpm)',   '#e63946'),
    ('E4_EDA',  'value', 'EDA (μS)',   '#2a9d8f'),
    ('E4_TEMP', 'value', 'Skin T (°C)','#e9c46a'),
    ('E4_BVP',  'value', 'BVP',        '#457b9d'),
]
for ax, (sig, val_col, label, color) in zip(axes, signal_info):
    df = e4_ex.get(sig)
    if df is not None and val_col in df.columns:
        ax.plot(df['t_sec'], df[val_col], color=color, linewidth=0.8, alpha=0.9)
    ax.set_ylabel(label, fontsize=9)
    ax2 = ax.twinx()
    if annot_ex is not None:
        ax2.plot(annot_ex['seconds'], annot_ex['arousal'], 'b--', linewidth=1.2,
                 alpha=0.5, label='arousal')
        ax2.set_ylim(0, 6); ax2.set_ylabel('Arousal', color='blue', fontsize=8)
        ax2.tick_params(axis='y', colors='blue')

axes[-1].set_xlabel('Time (s)')
plt.suptitle(f'P{pid_ex} — Physiological Signals + Arousal Annotations', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Audio & Video Features

In [ ]:
# Audio feature distributions
audio_labels = {
    'aud_F0semitoneFrom27.5Hz_sma3nz_amean': 'F0 mean',
    'aud_F0semitoneFrom27.5Hz_sma3nz_stddevNorm': 'F0 std',
    'aud_loudness_sma3_amean': 'Loudness mean',
    'aud_loudness_sma3_stddevNorm': 'Loudness std',
    'aud_jitterLocal_sma3nz_amean': 'Jitter',
    'aud_shimmerLocaldB_sma3nz_amean': 'Shimmer (dB)',
    'aud_HNRdBACF_sma3nz_amean': 'HNR (dB)',
}

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes_flat = axes.flatten()

for i, (col, label) in enumerate(audio_labels.items()):
    data = master[col].dropna()
    axes_flat[i].hist(data, bins=30, color='#DD8452', edgecolor='white', alpha=0.85)
    axes_flat[i].set_title(label)
    axes_flat[i].set_ylabel('Count')
    axes_flat[i].axvline(data.mean(), color='navy', linestyle='--',
                          label=f'μ={data.mean():.2f}')
    axes_flat[i].legend(fontsize=8)

axes_flat[-1].set_visible(False)
plt.suptitle('Audio Feature Distributions (eGeMAPS subset)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Video: face detection rate and AU distributions
vid_pids = sorted(vid._VIDEO_MAP.keys())

# Face detection rate per participant
det_rates = {}
for pid in vid_pids:
    cache = CACHE_VID / f'P{pid}.csv'
    if cache.exists():
        df = pd.read_csv(cache)
        det_rates[pid] = 1 - df['vid_au4_mean'].isna().mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
pids_sorted = sorted(det_rates.keys())
rates = [det_rates[p] for p in pids_sorted]
colors = ['#55A868' if r > 0.8 else '#C44E52' for r in rates]
ax.bar(range(len(pids_sorted)), rates, color=colors)
ax.set_xticks(range(len(pids_sorted)))
ax.set_xticklabels([f'P{p}' for p in pids_sorted], rotation=45, ha='right')
ax.axhline(0.8, color='black', linestyle='--', label='80% threshold')
ax.set_ylabel('Face detection rate'); ax.set_title('Face Detection Rate per Participant')
ax.legend()

# AU distribution comparison
ax = axes[1]
au_cols = [c for c in video_cols if c.endswith('_mean') and 'au' in c]
au_data = [master[c].dropna().values for c in au_cols]
au_labels = [c.replace('vid_','').replace('_mean','') for c in au_cols]
bp = ax.boxplot(au_data, labels=au_labels, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('#55A868'); patch.set_alpha(0.7)
ax.set_title('AU Feature Distributions (mean, all participants)')
ax.set_ylabel('AU proxy value')

plt.suptitle('Video Feature Analysis', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Feature–Target Correlations

In [ ]:
# Pearson correlation of every feature with arousal and valence
corr_rows = []
for col in feat_cols:
    sub = master[[col, 'arousal', 'valence']].dropna()
    if len(sub) < 30:
        continue
    for tgt in ['arousal', 'valence']:
        r, p = pearsonr(sub[col], sub[tgt])
        corr_rows.append({'feature': col, 'target': tgt,
                           'pearson_r': r, 'p_value': p,
                           'modality': ('audio' if col.startswith('aud_') else
                                        'video' if col.startswith('vid_') else 'physio')})

corr_df = pd.DataFrame(corr_rows)

# Top-15 absolute correlations per target
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
pal = {'physio':'#4C72B0', 'audio':'#DD8452', 'video':'#55A868'}

for ax, tgt in zip(axes, ['arousal','valence']):
    sub = corr_df[corr_df['target']==tgt].copy()
    sub['abs_r'] = sub['pearson_r'].abs()
    top = sub.nlargest(15, 'abs_r')
    colors = [pal[m] for m in top['modality']]
    bars = ax.barh(range(len(top)), top['pearson_r'].values,
                   color=colors, alpha=0.85)
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels([c.split('_sma3')[0].replace('aud_','').replace('e4_','').replace('vid_','')[:30]
                        for c in top['feature']], fontsize=9)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Pearson r')
    ax.set_title(f'Top-15 Feature–{tgt} Correlations')
    patches = [mpatches.Patch(color=v, label=k) for k,v in pal.items()]
    ax.legend(handles=patches, fontsize=8)

plt.suptitle('Feature–Target Pearson Correlations', fontsize=13)
plt.tight_layout()
plt.show()

print('\nSignificant features (|r| > 0.05, p < 0.05):')
sig = corr_df[(corr_df['pearson_r'].abs() > 0.05) & (corr_df['p_value'] < 0.05)]
print(sig.groupby('target').size())

In [ ]:
# Feature–feature correlation heatmap (select key features)
key_feats = [
    'e4_hr_mean', 'e4_eda_mean', 'e4_temp_mean', 'e4_bvp_rms', 'e4_acc_mean',
    'polar_hr_mean', 'attention_mean', 'meditation_mean',
    'aud_F0semitoneFrom27.5Hz_sma3nz_amean', 'aud_loudness_sma3_amean',
    'aud_jitterLocal_sma3nz_amean', 'aud_HNRdBACF_sma3nz_amean',
    'vid_au4_mean', 'vid_au6_mean', 'vid_au12_mean', 'vid_yaw_mean', 'vid_gaze_mean',
    'arousal', 'valence'
]
key_feats = [c for c in key_feats if c in master.columns]

corr_matrix = master[key_feats].corr(method='pearson')

nice_names = {
    'e4_hr_mean':'HR', 'e4_eda_mean':'EDA', 'e4_temp_mean':'Skin T',
    'e4_bvp_rms':'BVP rms', 'e4_acc_mean':'ACC',
    'polar_hr_mean':'Polar HR', 'attention_mean':'Attention', 'meditation_mean':'Meditation',
    'aud_F0semitoneFrom27.5Hz_sma3nz_amean':'F0', 'aud_loudness_sma3_amean':'Loudness',
    'aud_jitterLocal_sma3nz_amean':'Jitter', 'aud_HNRdBACF_sma3nz_amean':'HNR',
    'vid_au4_mean':'AU4', 'vid_au6_mean':'AU6', 'vid_au12_mean':'AU12',
    'vid_yaw_mean':'Yaw', 'vid_gaze_mean':'Gaze',
    'arousal':'Arousal', 'valence':'Valence'
}
corr_matrix.index = [nice_names.get(c, c) for c in corr_matrix.index]
corr_matrix.columns = corr_matrix.index

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(corr_matrix, mask=mask, ax=ax, annot=True, fmt='.2f',
            cmap='RdBu_r', vmin=-1, vmax=1, center=0,
            linewidths=0.3, annot_kws={'size':7})
ax.set_title('Feature–Feature Pearson Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Cross-Person Correlations

In [ ]:
# For each dyad: correlate B's key features with A's arousal/valence
key_physio = ['e4_hr_mean','e4_eda_mean','e4_temp_mean','polar_hr_mean',
              'attention_mean','aud_F0semitoneFrom27.5Hz_sma3nz_amean','vid_au4_mean']
key_physio = [c for c in key_physio if c in master.columns]

cross_rows = []
for pid_a, pid_b in DYADS:
    if pid_a not in seg_tables or pid_b not in seg_tables:
        continue
    da = seg_tables[pid_a].set_index('seconds')
    db = seg_tables[pid_b].set_index('seconds')
    common = sorted(set(da.index) & set(db.index))
    if len(common) < 20:
        continue
    da_c = da.loc[common]; db_c = db.loc[common]

    for direction, (speaker, listener) in [('B→A',(db_c,da_c)),('A→B',(da_c,db_c))]:
        for feat in key_physio:
            if feat not in speaker.columns:
                continue
            for tgt in ['arousal','valence']:
                sub = pd.concat([speaker[feat], listener[tgt]], axis=1).dropna()
                if len(sub) < 10:
                    continue
                r, p = pearsonr(sub.iloc[:,0], sub.iloc[:,1])
                cross_rows.append({'dyad':f'{pid_a}-{pid_b}', 'direction':direction,
                                    'feature':feat, 'target':tgt, 'r':r, 'p':p})

cross_df = pd.DataFrame(cross_rows)

# Average cross-person correlation per feature × target
avg_cross = cross_df.groupby(['feature','target','direction'])['r'].mean().unstack(['target','direction'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
feat_labels = [c.replace('aud_F0semitoneFrom27.5Hz_sma3nz_amean','F0')
                .replace('e4_','').replace('_mean','').replace('vid_','') for c in avg_cross.index]

for ax, tgt in zip(axes, ['arousal','valence']):
    if tgt not in avg_cross.columns.get_level_values(0):
        continue
    sub = avg_cross[tgt]
    x = np.arange(len(sub))
    w = 0.35
    for i, (direction, color) in enumerate([('B→A','#4C72B0'),('A→B','#DD8452')]):
        if direction in sub.columns:
            ax.bar(x + (i-0.5)*w, sub[direction].values, w,
                   label=direction, color=color, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(feat_labels, rotation=30, ha='right', fontsize=9)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel('Mean Pearson r (across dyads)')
    ax.set_title(f'Cross-Person Correlation → {tgt}')
    ax.legend()

plt.suptitle('B\'s Features Correlated with A\'s Emotions (avg. across 16 dyads)', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Lagged Cross-Correlation (key for H₂)

In [ ]:
# Cross-correlation at lags -3 to +5 segments (= -15s to +25s)
# Positive lag: B's feature at t-lag predicts A's emotion at t

def lagged_corr(feat_series, target_series, max_lag=5):
    """Pearson r at each lag (positive lag = feature leads target)."""
    results = {}
    n = min(len(feat_series), len(target_series))
    f = np.array(feat_series[:n], dtype=float)
    t = np.array(target_series[:n], dtype=float)
    for lag in range(-3, max_lag+1):
        if lag > 0:
            f_lag, t_lag = f[lag:], t[:-lag]
        elif lag < 0:
            f_lag, t_lag = f[:lag], t[-lag:]
        else:
            f_lag, t_lag = f, t
        mask = np.isfinite(f_lag) & np.isfinite(t_lag)
        if mask.sum() < 10:
            results[lag] = np.nan
        else:
            r, _ = pearsonr(f_lag[mask], t_lag[mask])
            results[lag] = r
    return results

lag_feats = ['e4_eda_mean','e4_temp_mean','polar_hr_mean',
             'aud_F0semitoneFrom27.5Hz_sma3nz_amean','vid_au4_mean']
lag_feats = [c for c in lag_feats if c in master.columns]

lag_results = {feat: {tgt: [] for tgt in ['arousal','valence']} for feat in lag_feats}

for pid_a, pid_b in DYADS:
    if pid_a not in seg_tables or pid_b not in seg_tables:
        continue
    da = seg_tables[pid_a].sort_values('seconds')
    db = seg_tables[pid_b].sort_values('seconds')
    common = sorted(set(da['seconds']) & set(db['seconds']))
    if len(common) < 20: continue
    da_c = da.set_index('seconds').loc[common]
    db_c = db.set_index('seconds').loc[common]

    for feat in lag_feats:
        if feat not in db_c.columns: continue
        for tgt in ['arousal','valence']:
            lc = lagged_corr(db_c[feat].values, da_c[tgt].values)
            lag_results[feat][tgt].append(lc)

lags = list(range(-3, 6))
fig, axes = plt.subplots(len(lag_feats), 2, figsize=(14, 3.5*len(lag_feats)))

for row, feat in enumerate(lag_feats):
    fname = (feat.replace('aud_F0semitoneFrom27.5Hz_sma3nz_amean','F0')
                 .replace('e4_','').replace('_mean','').replace('vid_',''))
    for col, tgt in enumerate(['arousal','valence']):
        ax = axes[row][col]
        session_data = lag_results[feat][tgt]
        mean_r = [np.nanmean([s[l] for s in session_data]) for l in lags]
        std_r  = [np.nanstd( [s[l] for s in session_data]) for l in lags]
        ax.bar(lags, mean_r, color=['#55A868' if r > 0 else '#C44E52' for r in mean_r],
               alpha=0.8, yerr=std_r, capsize=3)
        ax.axvline(0, color='black', linewidth=1, linestyle='--')
        ax.axhline(0, color='gray', linewidth=0.5)
        ax.set_xticks(lags)
        ax.set_xticklabels([f'{l*5}s' for l in lags])
        ax.set_title(f'{fname} → {tgt}', fontsize=10)
        ax.set_ylabel('Mean r (±SD)')
        # Mark best lag
        best_lag = lags[np.nanargmax(np.abs(mean_r))]
        ax.axvline(best_lag, color='blue', linewidth=1.5, alpha=0.5,
                   label=f'best lag={best_lag*5}s')
        ax.legend(fontsize=8)

plt.suptitle("Lagged Cross-Correlation: B's Feature → A's Emotion\n"
             "(positive lag = B's feature precedes A's annotation; supports H₂)",
             fontsize=12)
plt.tight_layout()
plt.show()

## 8. Dyadic Physiological Synchrony

In [ ]:
# Within-dyad: are partners' physiological signals correlated?
sync_feats = ['e4_hr_mean','e4_eda_mean','e4_temp_mean','polar_hr_mean','attention_mean']
sync_feats = [c for c in sync_feats if c in master.columns]

sync_rows = []
for pid_a, pid_b in DYADS:
    if pid_a not in seg_tables or pid_b not in seg_tables:
        continue
    da = seg_tables[pid_a].set_index('seconds')
    db = seg_tables[pid_b].set_index('seconds')
    common = sorted(set(da.index) & set(db.index))
    if len(common) < 20: continue

    for feat in sync_feats:
        if feat not in da.columns or feat not in db.columns: continue
        sub = pd.concat([da.loc[common, feat], db.loc[common, feat]], axis=1).dropna()
        if len(sub) < 10: continue
        r, p = pearsonr(sub.iloc[:,0], sub.iloc[:,1])
        sync_rows.append({'dyad':f'{pid_a}-{pid_b}','feature':feat,'r':r,'p':p})

sync_df = pd.DataFrame(sync_rows)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
sync_pivot = sync_df.pivot_table(index='dyad', columns='feature', values='r')
sync_pivot.columns = [c.replace('e4_','').replace('_mean','').replace('polar_hr','polar HR') for c in sync_pivot.columns]
sns.heatmap(sync_pivot, ax=ax, cmap='RdBu_r', center=0, vmin=-0.5, vmax=0.5,
            annot=True, fmt='.2f', linewidths=0.3, annot_kws={'size':8})
ax.set_title('Within-Dyad Physiological Synchrony (Pearson r)')

ax = axes[1]
avg_sync = sync_df.groupby('feature')['r'].agg(['mean','std']).sort_values('mean', ascending=False)
avg_sync.index = [c.replace('e4_','').replace('_mean','') for c in avg_sync.index]
ax.barh(avg_sync.index, avg_sync['mean'], xerr=avg_sync['std'],
        color=['#55A868' if v>0 else '#C44E52' for v in avg_sync['mean']],
        capsize=4, alpha=0.85)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Mean r (across dyads ± SD)')
ax.set_title('Average Within-Dyad Synchrony')

plt.suptitle('Physiological Synchrony Between Debate Partners', fontsize=13)
plt.tight_layout()
plt.show()

print('\nAverage synchrony by signal:')
print(avg_sync.round(3))

## 9. Summary Table

In [ ]:
# Comprehensive descriptive stats table
key_cols = ['e4_hr_mean','e4_eda_mean','e4_temp_mean','e4_ibi_mean',
            'polar_hr_mean','attention_mean','meditation_mean',
            'aud_F0semitoneFrom27.5Hz_sma3nz_amean','aud_loudness_sma3_amean',
            'aud_HNRdBACF_sma3nz_amean',
            'vid_au4_mean','vid_au12_mean','vid_gaze_mean',
            'arousal','valence']
key_cols = [c for c in key_cols if c in master.columns]

desc = master[key_cols].describe().T[['count','mean','std','min','50%','max']]
desc.columns = ['N','Mean','SD','Min','Median','Max']
desc['NaN%'] = (master[key_cols].isna().mean() * 100).round(1)
desc['r_arousal'] = [corr_df[(corr_df['feature']==c)&(corr_df['target']=='arousal')]['pearson_r'].values[0]
                     if len(corr_df[(corr_df['feature']==c)&(corr_df['target']=='arousal')]) else np.nan
                     for c in key_cols]
desc['r_valence'] = [corr_df[(corr_df['feature']==c)&(corr_df['target']=='valence')]['pearson_r'].values[0]
                     if len(corr_df[(corr_df['feature']==c)&(corr_df['target']=='valence')]) else np.nan
                     for c in key_cols]

desc.index = [c.replace('aud_F0semitoneFrom27.5Hz_sma3nz_amean','F0_mean').replace('aud_','').replace('e4_','').replace('vid_','') for c in desc.index]
print('\nDescriptive Statistics + Target Correlations:')
pd.set_option('display.float_format', '{:.3f}'.format)
display(desc)

In [ ]:
# Final takeaways
print('='*60)
print('EDA SUMMARY')
print('='*60)
print(f'Participants:          {len(seg_tables)}/32')
print(f'Usable dyads:          16/16')
print(f'Total segments:        {len(master)}')
print(f'Avg segments/session:  {len(master)/len(seg_tables):.0f}')
print()
print('Annotation ranges (external observer):')
for tgt in ['arousal','valence']:
    d = master[tgt]
    print(f'  {tgt}: mean={d.mean():.2f}  std={d.std():.2f}  range=[{d.min():.0f},{d.max():.0f}]')
print()
print('Inter-rater agreement (mean pairwise r):')
print(icc_df.groupby('target')['mean_pairwise_r'].mean().round(3).to_string())
print()
print('Strongest cross-person features (avg |r| with partner emotion):')
top_cross = cross_df.groupby('feature')['r'].apply(lambda x: x.abs().mean()).nlargest(5)
for f, r in top_cross.items():
    print(f'  {f.replace("aud_F0semitoneFrom27.5Hz_sma3nz_amean","F0").replace("e4_","").replace("vid_","").replace("_mean",""):25s}  |r|={r:.3f}')
print()
print('Strongest dyadic synchrony (same-signal within-dyad r):')
top_sync = avg_sync.nlargest(3, 'mean')
for f, row in top_sync.iterrows():
    print(f'  {f:20s}  r={row["mean"]:.3f} ± {row["std"]:.3f}')